In [1]:
import pandas as pd

# Load the 2010-2011 sheet 
df = pd.read_excel("online_retail_II.xlsx", sheet_name="Year 2010-2011")
raw_rows = len(df)
print(f"Raw rows: {raw_rows:,}")
df.head()

Raw rows: 541,910


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
# Cancelled orders will have invoice numbers starting with "C"
cancelled = df["Invoice"].astype(str).str.startswith("C")
print(f"Cancelled rows: {cancelled.sum():,}")
df = df[~cancelled]

# Removing rows with zero/negative quantity or price 
bad_qty_price = ((df["Quantity"] <= 0) | (df["Price"] <= 0)).sum()
print(f"Zero/negative quantity or price: {bad_qty_price:,}")
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]

# Removing rows with no customer ID 
missing_id = df["Customer ID"].isna().sum()
print(f"Missing customer ID: {missing_id:,}")
df = df.dropna(subset=["Customer ID"])

# Remove exact duplicate rows
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes:,}")
df = df.drop_duplicates()

# Tidy types 
df["Customer ID"] = df["Customer ID"].astype(int)
df["Revenue"] = df["Quantity"] * df["Price"]
df["Month"] = df["InvoiceDate"].dt.to_period("M").astype(str)

clean_rows = len(df)
print(f"\nKept {clean_rows:,} of {raw_rows:,} rows ({1 - clean_rows/raw_rows:.1%} removed)")

df.to_csv("basketiq_clean.csv", index=False)
print("Saved basketiq_clean.csv")

Cancelled rows: 9,288
Zero/negative quantity or price: 2,517
Missing customer ID: 132,220
Duplicate rows: 5,192

Kept 392,693 of 541,910 rows (27.5% removed)
Saved basketiq_clean.csv


In [11]:
import sqlite3

# Loading the cleaned data into an SQL database 
conn = sqlite3.connect(":memory:")
df.rename(columns={"Customer ID": "CustomerID"}).to_sql("sales", conn, index=False)

def q(sql):
    return pd.read_sql(sql, conn)

print("Database ready")

Database ready


In [12]:
# 1. Headline KPIs
display(q("""
SELECT ROUND(SUM(Revenue), 0)                          AS total_revenue,
       COUNT(DISTINCT Invoice)                          AS orders,
       COUNT(DISTINCT CustomerID)                       AS customers,
       ROUND(SUM(Revenue) / COUNT(DISTINCT Invoice), 2) AS avg_order_value
FROM sales
"""))

# 2. Monthly revenue
display(q("""
SELECT Month, ROUND(SUM(Revenue), 0) AS revenue, COUNT(DISTINCT Invoice) AS orders
FROM sales GROUP BY Month ORDER BY Month
"""))

# 3. Top 10 products 
display(q("""
SELECT Description, SUM(Quantity) AS units, ROUND(SUM(Revenue), 0) AS revenue
FROM sales
WHERE CAST(StockCode AS TEXT) NOT IN ('POST', 'DOT', 'M', 'BANK CHARGES', 'C2', 'AMAZONFEE')
GROUP BY Description ORDER BY revenue DESC LIMIT 10
"""))

# 4. Top 10 countries by revenue
display(q("""
SELECT Country, ROUND(SUM(Revenue), 0) AS revenue,
       ROUND(100.0 * SUM(Revenue) / (SELECT SUM(Revenue) FROM sales), 1) AS pct_of_revenue
FROM sales GROUP BY Country ORDER BY revenue DESC LIMIT 10
"""))

# 5. Repeat vs one-off customers
display(q("""
WITH c AS (
    SELECT CustomerID, COUNT(DISTINCT Invoice) AS orders, SUM(Revenue) AS rev
    FROM sales GROUP BY CustomerID
)
SELECT CASE WHEN orders = 1 THEN 'One-off' ELSE 'Repeat' END AS customer_type,
       COUNT(*) AS customers,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM c), 1)    AS pct_customers,
       ROUND(100.0 * SUM(rev) / (SELECT SUM(rev) FROM c), 1)    AS pct_revenue
FROM c GROUP BY customer_type
"""))

# 6. Share of revenue from the top 10% of customers
display(q("""
WITH c AS (SELECT CustomerID, SUM(Revenue) AS rev FROM sales GROUP BY CustomerID),
     d AS (SELECT rev, NTILE(10) OVER (ORDER BY rev DESC) AS decile FROM c)
SELECT ROUND(100.0 * SUM(CASE WHEN decile = 1 THEN rev END) / SUM(rev), 1)
       AS top_10pct_customers_revenue_share
FROM d
"""))

,total_revenue,orders,customers,avg_order_value
0,8602604.0,18529,4337,464.28


,Month,revenue,orders
0,2010-12,570423.0,1400
1,2011-01,490918.0,986
2,2011-02,446085.0,997
3,2011-03,594082.0,1321
4,2011-04,468374.0,1149
5,2011-05,677355.0,1555
6,2011-06,621076.0,1392
7,2011-07,598963.0,1331
8,2011-08,644051.0,1280
9,2011-09,950690.0,1755


,Description,units,revenue
0,REGENCY CAKESTAND 3 TIER,12374,142265.0
1,WHITE HANGING HEART T-LIGHT HOLDER,36706,100392.0
2,JUMBO BAG RED RETROSPOT,46078,85041.0
3,PARTY BUNTING,15279,68785.0
4,ASSORTED COLOUR BIRD ORNAMENT,35263,56413.0
5,RABBIT NIGHT LIGHT,27153,51251.0
6,CHILLI LIGHTS,9646,46265.0
7,PAPER CHAIN KIT 50'S CHRISTMAS,15591,42584.0
8,BLACK RECORD COVER FRAME,11401,39046.0
9,JUMBO BAG PINK POLKADOT,20148,37254.0


,Country,revenue,pct_of_revenue
0,United Kingdom,7000401.0,81.4
1,Netherlands,285446.0,3.3
2,EIRE,265262.0,3.1
3,Germany,228678.0,2.7
4,France,208952.0,2.4
5,Australia,138454.0,1.6
6,Spain,61559.0,0.7
7,Switzerland,56444.0,0.7
8,Belgium,41196.0,0.5
9,Sweden,38368.0,0.4


,customer_type,customers,pct_customers,pct_revenue
0,One-off,1493,34.4,6.2
1,Repeat,2844,65.6,93.8


,top_10pct_customers_revenue_share
0,60.3


In [5]:
df.nlargest(5, "Quantity")[["Invoice", "Description", "Quantity", "InvoiceDate", "Customer ID"]]

,Invoice,Description,Quantity,InvoiceDate,Customer ID
540421,581483,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,16446
61619,541431,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,12346
421632,573008,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,2011-10-27 12:26:00,12901
206121,554868,SMALL POPCORN HOLDER,4300,2011-05-27 10:52:00,13135
97432,544612,EMPIRE DESIGN ROSETTE,3906,2011-02-22 10:43:00,18087


In [6]:
# Removing the two bulk orders that were cancelled immediately 
before = len(df)
df = df[df["Quantity"] < 70000]
print(f"Removed {before - len(df)} outlier rows")
df.to_csv("basketiq_clean.csv", index=False)

Removed 2 outlier rows


In [9]:
df[df["Description"].str.contains("PICNIC BASKET WICKER 60", na=False)]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Month
222671,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,2011-06-10 15:28:00,649.5,15098,United Kingdom,38970.0,2011-06
222682,556446,22502,PICNIC BASKET WICKER 60 PIECES,1,2011-06-10 15:33:00,649.5,15098,United Kingdom,649.5,2011-06


In [10]:
   df = df[df["Invoice"].astype(str) != "556444"]
   df.to_csv("basketiq_clean.csv", index=False)
   print(f"Rows now: {len(df):,}")

Rows now: 392,690
